# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

**Group / repo name:** `aidams-lab1-<surname1>-<surname2>-...`  
**Submitter (one person):**  
**Repo URL:**  
**Streamlit Cloud URL (bonus):**  

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [1]:
# Import required libraries
# - pandas for data manipulation
# - numpy for numerical operations
# - plotly.express and plotly.graph_objects for interactive visualizations
# - Any other libraries you need
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
import os


In [2]:
# Load the steel plants dataset
# Locate the plant-level steel dataset
steel_folder = Path("Steel Datasets")

plant_files = list(
    steel_folder.glob("Plant-level_data_Global_Iron_and_Steel_Tracker*.xlsx")
)

file_path = plant_files[0]

df = pd.read_excel(file_path, sheet_name="Plant data")

# Quick check
df.head()
# Tip: start with df.columns / df.head() and adapt names if your file differs slightly
#
# Common columns in this steel plant dataset:
# - Plant name (English)
# - Owner
# - Country/Area, Region
# - Coordinates  (often a single "lat, lon" string — not separate latitude/longitude columns)
# - Plant age (years)
# - Capacity field: usually "Nominal crude steel capacity (ttpa)" (check your df.columns to confirm the exact name)
#   (Older datasets may also include additional fields like ferronickel/sinter/coking/pelletizing capacities, 
#    but most analyses focus on nominal crude steel capacity as the main output.)


,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Steel products,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,"billet, wire rod, angle, flat, bar, square bar...",unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,"billet, wire rod, rebar",building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,"wire rod, rebar, bar, billet, round bar, wire",unknown,4500,2025-10-06 00:00:00,unknown,no,EAF,unknown,NaN,unknown
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"billet, rebar","building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,"pipe, tube, flat","automotive, building and infrastructure, energ...",11000,2025-04-30 00:00:00,2025-12-04 00:00:00,no,DRI; EAF; BF; BOF,unknown,unknown,unknown


---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [3]:
# Display dataset shape
print("Dataset shape:", df.shape)
print("Number of steel plants:", df.shape[0])


Dataset shape: (1293, 44)
Number of steel plants: 1293


In [4]:
# Display column information and data types
# Start here: print(df.columns) and adapt column names in later cells if needed
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Column names:
['GEM plant ID', 'Plant name (English)', 'Plant name (other language)', 'Other plant names (English)', 'Other plant names (other language)', 'Owner', 'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status', 'Parent (English)', 'Parent GEM entity ID', 'Parent PermID', 'Location address', 'Location address (other language)', 'Municipality', 'Subnational unit', 'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy', 'GEM wiki page', 'Plant age', 'Announced date', 'Construction date', 'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date', 'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)', 'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products', 'Steel sector end users', 'Workforce size', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification', 'Main production equipment', 'Power source', 'Iron ore source', 'Met coal source']

Data types:
GEM plant ID  

In [5]:
# Check for missing values
missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)

print("\nTotal missing values in dataset:", missing_values.sum())


Missing values by column:
GEM plant ID                             0
Plant name (English)                     0
Plant name (other language)            502
Other plant names (English)            551
Other plant names (other language)     958
Owner                                    0
Owner (other language)                 715
Owner GEM entity ID                      0
Owner PermID                             0
SOE status                            1081
Parent (English)                         0
Parent GEM entity ID                     0
Parent PermID                            0
Location address                         0
Location address (other language)      794
Municipality                             0
Subnational unit                         0
Country/area                             0
Region                                   0
Coordinates                              0
Coordinate accuracy                      0
GEM wiki page                            0
Plant age                   

### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [6]:
# Columns available in plant dataset
capacity_cols = [
    "Ferronickel capacity (ttpa)",
    "Sinter plant capacity (ttpa)",
    "Coking plant capacity (ttpa)",
    "Pelletizing plant capacity (ttpa)"
]

# Convert specific columns to numeric
for col in capacity_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Plant age"] = pd.to_numeric(df["Plant age"], errors="coerce")

# Calculate total capacity per plant
df["Total capacity (ttpa)"] = df[capacity_cols].sum(axis=1, min_count=1)

# Parse Coords into Lat and Long
coords = df["Coordinates"].str.split(",", expand=True)

df["Latitude"] = pd.to_numeric(coords[0], errors="coerce")
df["Longitude"] = pd.to_numeric(coords[1], errors="coerce")

# Display descriptive statistics
print("Average plant capacity (ttpa):",
      df["Total capacity (ttpa)"].mean())

print("\nLatitude range:")
print(df["Latitude"].min(), "to", df["Latitude"].max())

print("\nLongitude range:")
print(df["Longitude"].min(), "to", df["Longitude"].max())

print("\nPlant age distribution:")
print(df["Plant age"].describe())

print("\nDescriptive statistics:")
df[["Total capacity (ttpa)", "Latitude", "Longitude", "Plant age"]].describe()


Average plant capacity (ttpa): 5730.9625468164795

Latitude range:
-37.831379 to 67.189096

Longitude range:
-123.163599 to 174.728098

Plant age distribution:
count    1125.00000
mean       38.78200
std        36.42716
min         0.00000
25%        16.00000
50%        25.00000
75%        55.00000
max       287.00000
Name: Plant age, dtype: float64

Descriptive statistics:


,Total capacity (ttpa),Latitude,Longitude,Plant age
count,267.000000,1293.000000,1293.000000,1125.00000
mean,5730.962547,30.107078,64.225291,38.78200
std,6365.563142,16.678049,66.403833,36.42716
min,55.000000,-37.831379,-123.163599,0.00000
25%,1500.000000,23.504558,27.137563,16.00000
50%,3700.000000,33.962272,87.295932,25.00000
75%,7379.000000,39.976702,115.125838,55.00000
max,42670.000000,67.189096,174.728098,287.00000


### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [7]:
# Count plants by country/region
print("Top 15 countries/areas by number of steel plants:")
country_counts = df["Country/area"].value_counts()
print(country_counts.head(15))

print("\nPlants by region:")
region_counts = df["Region"].value_counts()
print(region_counts)


Top 15 countries/areas by number of steel plants:
Country/area
China            458
India            113
United States     90
Iran              56
Japan             42
Russia            31
Türkiye           30
Vietnam           28
Brazil            25
Italy             24
Germany           19
South Korea       18
Spain             17
Malaysia          16
Indonesia         16
Name: count, dtype: int64

Plants by region:
Region
Asia Pacific               765
Europe                     184
North America              113
Middle East                 90
Africa                      51
Eurasia                     47
Central & South America     43
Name: count, dtype: int64


In [8]:
# Count plants by Owner (company)
owner_counts = df["Owner"].value_counts()

print("Top 20 owners by number of steel plants:")
print(owner_counts.head(20))

Top 20 owners by number of steel plants:
Owner
Nucor Corp                              13
Cleveland-Cliffs Inc                    12
Nippon Steel Corp                       10
Commercial Metals Co                     8
Gerdau Ameristeel Corp                   8
Steel Authority of India Ltd             8
SteelAsia Manufacturing Corp             7
ArcelorMittal Brasil SA                  6
ArcelorMittal SA                         6
Liberty Steel Group                      6
Steel Dynamics Inc                       6
Tata Steel Ltd                           6
United States Steel Corp                 6
ArcelorMittal Nippon Steel India Ltd     5
JFE Steel Corp                           5
Jindal Steel Limited Ltd                 5
JSW Steel Ltd                            5
Rungta Mines Ltd                         5
Government of North Korea                4
Gerdau Acos Longos SA                    4
Name: count, dtype: int64


### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?


In [9]:
# Calculate total capacity
total_global_capacity = df["Total capacity (ttpa)"].sum()

print(f"Total global capacity (available fields): {total_global_capacity:,.0f} ttpa")

print("\nTotal capacity by region:")
capacity_by_region = (
    df.groupby("Region")["Total capacity (ttpa)"]
    .sum()
    .sort_values(ascending=False)
)
print(capacity_by_region)

print("\nTop 15 countries/areas by total capacity:")
capacity_by_country = (
    df.groupby("Country/area")["Total capacity (ttpa)"]
    .sum()
    .sort_values(ascending=False)
)
print(capacity_by_country.head(15))


Total global capacity (available fields): 1,530,167 ttpa

Total capacity by region:
Region
Asia Pacific               1033993.0
Eurasia                     147694.0
Europe                      121884.0
Middle East                  97585.0
Central & South America      58415.0
Africa                       42985.0
North America                27611.0
Name: Total capacity (ttpa), dtype: float64

Top 15 countries/areas by total capacity:
Country/area
India            469533.0
China            445121.0
Russia           133948.0
Iran              82785.0
Brazil            53170.0
Japan             37445.0
Ukraine           37044.0
Australia         30700.0
Germany           28580.0
France            24530.0
United States     18377.0
Vietnam           18211.0
South Korea       13230.0
Bahrain           12000.0
Netherlands       11400.0
Name: Total capacity (ttpa), dtype: float64


In [10]:
# Group by Owner and sum capacity
capacity_by_owner = (
    df.groupby("Owner")["Total capacity (ttpa)"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)

capacity_by_owner.columns = [
    "Total capacity (ttpa)",
    "Average capacity (ttpa)",
    "Number of plants with capacity data"
]

print("Top 20 owners by total capacity:")
print(capacity_by_owner.head(20))

Top 20 owners by total capacity:
                                                    Total capacity (ttpa)  \
Owner                                                                       
JSW Steel Ltd                                                     50935.0   
Steel Authority of India Ltd                                      43591.0   
Jindal Steel Odisha Ltd                                           42670.0   
Jindal Steel Limited Ltd                                          39710.0   
Rungta Mines Ltd                                                  34955.0   
Tata Steel Ltd                                                    33787.0   
Lebedinskiy GOK JSC                                               31031.0   
Progressive Green Solutions JV                                    30000.0   
JSW Jharkhand Steel Ltd                                           29600.0   
Maanshan Iron & Steel Co Ltd                                      25540.0   
JSW Utkal Steel Ltd                        

---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [11]:
# Create a scatter_geo or scatter_mapbox plot
# Hint: Use plotly.express.scatter_geo() or scatter_mapbox()
#
# Coordinates hint:
#   The dataset usually stores location in a single Coordinates column like "lat, lon".
#   Split it before plotting

# Keeps only rows with valid coordinates
map_df = df.dropna(subset=["Latitude", "Longitude"]).copy()

# Creates the map
fig = px.scatter_geo(
    map_df,
    lat="Latitude",
    lon="Longitude",
    hover_name="Plant name (English)",
    hover_data=["Country/area", "Owner"],
    title="Global Distribution of Steel Plants"
)

fig.update_geos(
    showland=True,
    showcountries=True,
    showcoastlines=True
)

fig.show()

### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [24]:
# Create scatter map with size parameter based on capacity
# Keeps plants with valid coordinates and capacity data
capacity_map_df = df.dropna(
    subset=["Latitude", "Longitude", "Total capacity (ttpa)"]
).copy()

# Creates map
fig_capacity = px.scatter_geo(
    capacity_map_df,
    lat="Latitude",
    lon="Longitude",
    size="Total capacity (ttpa)",
    color="Owner",
    hover_name="Plant name (English)",
    hover_data={
        "Country/area": True,
        "Owner": True,
        "Total capacity (ttpa)": ":,.0f",
        "Latitude": False,
        "Longitude": False
    },
    title="Global Steel Plants by Capacity",
    size_max=35
)

fig_capacity.update_geos(
    showland=True,
    showcountries=True,
    showcoastlines=True
)

fig_capacity.show()


### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [26]:
# Create density heatmap
# Hint: Use plotly.express.density_mapbox()
# Keep only plants with valid coordinates
density_df = df.dropna(subset=["Latitude", "Longitude"]).copy()

# Create heatmap
fig_density = px.density_map(
    density_df,
    lat="Latitude",
    lon="Longitude",
    radius=10,
    zoom=1,
    center={"lat": 20, "lon": 20},
    map_style="open-street-map",
    title="Global Density of Steel Plants"
)

fig_density.show()


---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.


### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [28]:
# Load LitPop sample (exposure / population–asset data)
from pathlib import Path
import h5py

# Locates files
litpop_folder = Path("litpop")
litpop_files = list(litpop_folder.glob("*.hdf5"))

print("LitPop files found:")
for file in litpop_files:
    print(file)

# Inspects structure of first file
with h5py.File(litpop_files[0], "r") as f:
    print("\nContents of first HDF5 file:")
    print(list(f.keys()))

LitPop files found:
litpop\LitPop_pc_300_arcsec_CHN_v1.hdf5
litpop\LitPop_pc_300_arcsec_IND_v1.hdf5
litpop\LitPop_pc_300_arcsec_JPN_v1.hdf5

Contents of first HDF5 file:
['exposures']


In [32]:
# Inspect LitPop data (columns, dtypes, missing values, value ranges)
litpop_dfs = []

for file in litpop_files:
    temp = pd.read_hdf(file, key="exposures")

    # Add country code 
    country_code = file.stem.split("_")[-2]
    temp["Country_code"] = country_code

    litpop_dfs.append(temp)

# Combine China, India, Japan
litpop_df = pd.concat(litpop_dfs, ignore_index=True)

print("LitPop shape:", litpop_df.shape)

print("\nColumns:")
print(litpop_df.columns.tolist())

print("\nData types:")
print(litpop_df.dtypes)

print("\nMissing values:")
print(litpop_df.isnull().sum())

print("\nFirst 5 rows:")
display(litpop_df.head())

print("\nNumerical summary:")
display(litpop_df.describe())

LitPop shape: (182591, 7)

Columns:
['value', 'latitude', 'longitude', 'geometry', 'region_id', 'impf_', 'Country_code']

Data types:
value           float64
latitude        float64
longitude       float64
geometry         object
region_id         int64
impf_             int64
Country_code        str
dtype: object

Missing values:
value           0
latitude        0
longitude       0
geometry        0
region_id       0
impf_           0
Country_code    0
dtype: int64

First 5 rows:


,value,latitude,longitude,geometry,region_id,impf_,Country_code
0,5.280440e+09,20.041667,110.208333,POINT (110.20833333 20.04166667),156,1,CHN
1,4.040559e+07,20.041667,110.625000,POINT (110.625 20.04166667),156,1,CHN
2,4.190224e+07,20.041667,110.708333,POINT (110.70833333 20.04166667),156,1,CHN
3,8.813872e+07,19.958333,109.541667,POINT (109.54166667 19.95833333),156,1,CHN
4,1.879947e+08,19.958333,109.625000,POINT (109.625 19.95833333),156,1,CHN



Numerical summary:


,value,latitude,longitude,region_id,impf_
count,1.825910e+05,182591.000000,182591.000000,182591.000000,182591.0
mean,3.817856e+08,33.585715,99.540705,207.031891,1.0
std,5.670982e+09,8.816306,17.568848,88.645570,0.0
min,0.000000e+00,6.875000,68.208333,156.000000,1.0
25%,5.023580e+04,27.041667,83.875000,156.000000,1.0
50%,1.295843e+06,33.875000,98.708333,156.000000,1.0
75%,1.401763e+07,40.375000,113.875000,156.000000,1.0
max,5.044057e+11,53.541667,145.791667,392.000000,1.0


### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [36]:
# Calculate distances or perform spatial join
# Hint: You might calculate haversine distance or use a spatial library
from scipy.spatial import cKDTree

# Countries covered
country_map = {
    "China": "CHN",
    "India": "IND",
    "Japan": "JPN"
}
# Keep only plants in China, India, and Japan
plants_match = df[df["Country/area"].isin(country_map.keys())].copy()

# Add matching country code
plants_match["Country_code"] = plants_match["Country/area"].map(country_map)

# Keep valid coordinates
plants_match = plants_match.dropna(subset=["Latitude", "Longitude"])

matched_parts = []

# Match within each country separately
for country_code in ["CHN", "IND", "JPN"]:

    plants_country = plants_match[
        plants_match["Country_code"] == country_code
    ].copy()

    litpop_country = litpop_df[
        litpop_df["Country_code"] == country_code
    ].copy()

    tree = cKDTree(
        litpop_country[["latitude", "longitude"]].values
    )

    distances, indices = tree.query(
        plants_country[["Latitude", "Longitude"]].values,
        k=1
    )

    plants_country["distance_degrees"] = distances

    # Store actual dataframe index of nearest point
    plants_country["litpop_index"] = (
        litpop_country.iloc[indices].index.values
    )

    matched_parts.append(plants_country)

# Combine results
plants_match = pd.concat(matched_parts, ignore_index=True)

print("Plants matched:", len(plants_match))

print("\nPlants matched by country:")
print(plants_match["Country/area"].value_counts())

print("\nDistance summary:")
print(plants_match["distance_degrees"].describe())

print("\nFirst matches:")
display(
    plants_match[
        [
            "Plant name (English)",
            "Country/area",
            "Latitude",
            "Longitude",
            "litpop_index",
            "distance_degrees"
        ]
    ].head()
)

Plants matched: 613

Plants matched by country:
Country/area
China    458
India    113
Japan     42
Name: count, dtype: int64

Distance summary:
count    613.000000
mean       0.033352
std        0.015149
min        0.000333
25%        0.022832
50%        0.034510
75%        0.042089
max        0.164361
Name: distance_degrees, dtype: float64

First matches:


,Plant name (English),Country/area,Latitude,Longitude,litpop_index,distance_degrees
0,Angang Group Xinyang Iron and Steel Co Ltd,China,32.488704,114.038243,95076,0.030563
1,Angang Lianzhong Stainless Steel Co Ltd,China,23.126822,113.505817,134526,0.035896
2,Angang Steel Co Ltd,China,41.148267,122.983054,39980,0.033948
3,Angang Steel Co Ltd Bayuquan branch,China,40.308494,122.130790,45754,0.017796
4,Anhui Changjiang Steel Co Ltd,China,31.502272,118.465162,100761,0.039982


In [37]:
# Merge datasets
# Select the values corresponding to each matched plant
nearest_litpop = litpop_df.loc[
    plants_match["litpop_index"],
    ["value", "latitude", "longitude", "region_id", "impf_"]
].reset_index(drop=True)

# Rename columns
nearest_litpop = nearest_litpop.rename(columns={
    "value": "LitPop_value",
    "latitude": "LitPop_latitude",
    "longitude": "LitPop_longitude",
    "region_id": "LitPop_region_id",
    "impf_": "LitPop_impf"
})

# Combine plant data with its nearest LitPop data
merged_df = pd.concat(
    [plants_match.reset_index(drop=True), nearest_litpop],
    axis=1
)

print("Merged dataset shape:", merged_df.shape)
print("Number of plants:", len(merged_df))

print("\nMissing LitPop values:")
print(merged_df["LitPop_value"].isnull().sum())

print("\nFirst 5 merged rows:")
display(
    merged_df[
        [
            "Plant name (English)",
            "Country/area",
            "Latitude",
            "Longitude",
            "LitPop_value",
            "LitPop_latitude",
            "LitPop_longitude",
            "distance_degrees"
        ]
    ].head()
)


Merged dataset shape: (613, 55)
Number of plants: 613

Missing LitPop values:
0

First 5 merged rows:


,Plant name (English),Country/area,Latitude,Longitude,LitPop_value,LitPop_latitude,LitPop_longitude,distance_degrees
0,Angang Group Xinyang Iron and Steel Co Ltd,China,32.488704,114.038243,3.822503e+08,32.458333,114.041667,0.030563
1,Angang Lianzhong Stainless Steel Co Ltd,China,23.126822,113.505817,2.360539e+10,23.125000,113.541667,0.035896
2,Angang Steel Co Ltd,China,41.148267,122.983054,4.442241e+10,41.125000,122.958333,0.033948
3,Angang Steel Co Ltd Bayuquan branch,China,40.308494,122.130790,6.209927e+09,40.291667,122.125000,0.017796
4,Anhui Changjiang Steel Co Ltd,China,31.502272,118.465162,2.369022e+09,31.541667,118.458333,0.039982


### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [39]:
# Create visualization of plants colored by LitPop exposure metrics
# Keep rows with capacity data so marker size can be calculated
exposure_map_df = merged_df.dropna(
    subset=["Latitude", "Longitude", "Total capacity (ttpa)", "LitPop_value"]
).copy()

# Create map
fig_exposure = px.scatter_geo(
    exposure_map_df,
    lat="Latitude",
    lon="Longitude",
    size="Total capacity (ttpa)",
    color="LitPop_value",
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True,
        "Country/area": True,
        "Total capacity (ttpa)": ":,.0f",
        "LitPop_value": ":,.0f",
        "Latitude": False,
        "Longitude": False
    },
    size_max=35,
    title="Steel Plants with LitPop Exposure Context"
)

fig_exposure.update_geos(
    showland=True,
    showcountries=True,
    showcoastlines=True,
    fitbounds="locations"
)

fig_exposure.show()

Interpretation: Most of the large steel plants are located in China and India. Some of the larger plants, especially in eastern China and Japan, are also in areas with high LitPop exposure. Overall, this shows that many major steel plants are located near areas with high population or asset value.

---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [40]:
# Group by company and aggregate
company_summary = (
    merged_df
    .dropna(subset=["Owner"])
    .groupby("Owner")
    .agg(
        total_capacity=("Total capacity (ttpa)", "sum"),
        number_of_plants=("Plant name (English)", "count"),
        average_litpop=("LitPop_value", "mean"),
        number_of_countries=("Country/area", "nunique"),
        number_of_regions=("Region", "nunique")
    )
    .sort_values("total_capacity", ascending=False)
    .reset_index()
)

# Display results
print("Number of companies:", len(company_summary))

print("\nTop companies by total capacity:")
display(company_summary.head(20))


Number of companies: 535

Top companies by total capacity:


,Owner,total_capacity,number_of_plants,average_litpop,number_of_countries,number_of_regions
0,JSW Steel Ltd,50935.0,5,3.905037e+08,1,1
1,Steel Authority of India Ltd,43591.0,8,3.249197e+09,1,1
2,Jindal Steel Odisha Ltd,42670.0,1,3.228271e+08,1,1
3,Jindal Steel Limited Ltd,39710.0,5,3.743142e+08,1,1
4,Rungta Mines Ltd,34955.0,5,3.076285e+08,1,1
5,Tata Steel Ltd,33787.0,5,1.952889e+09,1,1
6,JSW Jharkhand Steel Ltd,29600.0,1,2.208526e+07,1,1
7,Maanshan Iron & Steel Co Ltd,25540.0,1,1.416193e+10,1,1
8,JSW Utkal Steel Ltd,24775.0,1,1.051607e+08,1,1
9,Nippon Steel Corp,24078.0,10,1.366037e+10,1,1


### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [41]:
# Calculate company representative locations
company_locations = (
    merged_df
    .dropna(subset=["Owner", "Latitude", "Longitude"])
    .groupby("Owner")
    .agg(
        representative_latitude=("Latitude", "mean"),
        representative_longitude=("Longitude", "mean"),
        number_of_plants=("Plant name (English)", "count")
    )
    .reset_index()
)

print("Company representative locations:", len(company_locations))

print("\nFirst 10 companies:")
display(company_locations.head(10))

# Option 1 was used as it's straightforward and actually represents the company's overall plant footprint rather than arbitrarily picking one plant


Company representative locations: 535

First 10 companies:


,Owner,representative_latitude,representative_longitude,number_of_plants
0,Action Ispat and Power Pvt Ltd,21.839759,83.982227,1
1,Adhunik Metaliks Ltd,22.308371,84.763584,1
2,Aichi Steel Corp,35.044558,136.900772,1
3,Angang Lianzhong Stainless Steel Corp,23.126822,113.505817,1
4,Angang Steel Co Ltd,40.995235,121.827942,3
5,Anhui Changjiang Steel Co Ltd,31.502272,118.465162,1
6,Anhui Guihang Special Steel Co Ltd,30.531068,117.251147,1
7,Anhui Jin'an Stainless Steel Foundry Co Ltd,31.763956,115.924372,1
8,Anhui Jingxian Longxin Iron and Steel Co Ltd,30.709058,118.436638,1
9,Anhui Langxi County Fuhang Steel Co Ltd,31.246635,119.120443,1


### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [42]:
# Create company-level visualization
# Combine company metrics with representative locations
company_map = company_summary.merge(
    company_locations[
        ["Owner", "representative_latitude", "representative_longitude"]
    ],
    on="Owner",
    how="left"
)

# Keep rows with values needed
company_map = company_map.dropna(
    subset=[
        "representative_latitude",
        "representative_longitude",
        "total_capacity",
        "average_litpop"
    ]
).copy()

# Create map
fig_company = px.scatter_geo(
    company_map,
    lat="representative_latitude",
    lon="representative_longitude",
    size="total_capacity",
    color="average_litpop",
    hover_name="Owner",
    hover_data={
        "total_capacity": ":,.0f",
        "number_of_plants": True,
        "average_litpop": ":,.0f",
        "number_of_countries": True,
        "number_of_regions": True,
        "representative_latitude": False,
        "representative_longitude": False
    },
    size_max=35,
    title="Company-Level Steel Capacity and LitPop Exposure"
)

fig_company.update_geos(
    showland=True,
    showcountries=True,
    showcoastlines=True,
    fitbounds="locations"
)

fig_company.show()


Interpretation: Most companies are concentrated in China and India, with some in Japan. Some of the biggest companies are also located in these areas. However, having a higher steel capacity does not always mean that the company is in an area with higher LitPop exposure.

---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading


In [43]:
# Save processed datasets
from pathlib import Path

# Create folder for dashboard data
output_folder = Path("dashboard_data")
output_folder.mkdir(exist_ok=True)

# Cleaned plant data
df.to_csv(output_folder / "plants_cleaned.csv", index=False)

# Plant data merged with LitPop exposure
merged_df.to_csv(output_folder / "plants_litpop.csv", index=False)

# Company-level aggregated data
company_summary.to_csv(
    output_folder / "company_summary.csv",
    index=False
)

# Company representative locations
company_locations.to_csv(
    output_folder / "company_locations.csv",
    index=False
)

print("Dashboard datasets saved successfully:")
for file in output_folder.glob("*.csv"):
    print("-", file)



Dashboard datasets saved successfully:
- dashboard_data\company_locations.csv
- dashboard_data\company_summary.csv
- dashboard_data\plants_cleaned.csv
- dashboard_data\plants_litpop.csv


### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

In [ ]:
# This cell is for notes/observations about your dashboard



# What works well?

# The dashboard makes it easy to see where the steel plants are located and compare
# their capacity and LitPop exposure. The filters also make it easier to focus on
# specific companies or countries instead of looking at everything at once.



# What could be improved?

# The map could be improved by adding more exposure metrics and giving the user more
# options for how the data is displayed. The company-level data could also be shown
# more clearly with additional graphs or rankings.



# Any performance issues with large datasets?

# The dashboard works well with the current dataset, but with a much larger dataset
# the interactive map and filtering could become slower. Saving the processed data
# beforehand helps since the dashboard does not have to redo the full analysis each time.



---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
